In [15]:
"""
Train a Chronic Kidney Disease prediction model.

Dataset: UCI Chronic Kidney Disease dataset (400 patients, Tamil Nadu, India)
Source (raw csv): ArjunAnilPillai/Chronic-Kidney-Disease-dataset on GitHub

We keep only the NUMERIC clinical features so that the Streamlit form stays
consistent with the rest of the app (plain st.text_input boxes, no dropdowns):

    age  - Age (years)
    bp   - Blood Pressure (mm/Hg)
    sg   - Specific Gravity
    al   - Albumin (0-5)
    su   - Sugar (0-5)
    bgr  - Blood Glucose Random (mgs/dl)
    bu   - Blood Urea (mgs/dl)
    sc   - Serum Creatinine (mgs/dl)
    sod  - Sodium (mEq/L)
    pot  - Potassium (mEq/L)
    hemo - Hemoglobin (gms)
    pcv  - Packed Cell Volume
    wc   - White Blood Cell Count (cells/cumm)
    rc   - Red Blood Cell Count (millions/cmm)

Target: class -> ckd = 1, notckd = 0
"""

'\nTrain a Chronic Kidney Disease prediction model.\n\nDataset: UCI Chronic Kidney Disease dataset (400 patients, Tamil Nadu, India)\nSource (raw csv): ArjunAnilPillai/Chronic-Kidney-Disease-dataset on GitHub\n\nWe keep only the NUMERIC clinical features so that the Streamlit form stays\nconsistent with the rest of the app (plain st.text_input boxes, no dropdowns):\n\n    age  - Age (years)\n    bp   - Blood Pressure (mm/Hg)\n    sg   - Specific Gravity\n    al   - Albumin (0-5)\n    su   - Sugar (0-5)\n    bgr  - Blood Glucose Random (mgs/dl)\n    bu   - Blood Urea (mgs/dl)\n    sc   - Serum Creatinine (mgs/dl)\n    sod  - Sodium (mEq/L)\n    pot  - Potassium (mEq/L)\n    hemo - Hemoglobin (gms)\n    pcv  - Packed Cell Volume\n    wc   - White Blood Cell Count (cells/cumm)\n    rc   - Red Blood Cell Count (millions/cmm)\n\nTarget: class -> ckd = 1, notckd = 0\n'

In [16]:
import pickle
import numpy as np
import pandas as pd
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [17]:
# ---------------------------------------------------------------------------
# Load & clean data
# ---------------------------------------------------------------------------
df = pd.read_csv('kidney_disease.csv')

In [18]:
numeric_cols = ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc',
                'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc']

In [19]:
# some columns (pcv, wc, rc) contain stray whitespace / non-numeric strings
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [20]:
# clean up target labels (some rows have trailing tabs/spaces e.g. "ckd\t")
df['class'] = df['class'].astype(str).str.strip()
df['class'] = df['class'].map({'ckd': 1, 'notckd': 0})

In [21]:
# drop rows where target itself is missing
df = df.dropna(subset=['class'])

In [22]:
# impute missing numeric values with the column median (simple + robust)
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

In [23]:
X = df[numeric_cols]
Y = df['class'].astype(int)

In [24]:
# ---------------------------------------------------------------------------
# Train / test split
# ---------------------------------------------------------------------------
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, stratify=Y, random_state=2
)

In [25]:
# ---------------------------------------------------------------------------
# Train model (linear SVM, consistent with the rest of the project)
# ---------------------------------------------------------------------------
classifier = svm.SVC(kernel='linear')
classifier.fit(X_train, Y_train)

SVC(kernel='linear')

In [26]:
train_acc = accuracy_score(Y_train, classifier.predict(X_train))
test_acc = accuracy_score(Y_test, classifier.predict(X_test))
print('Kidney Disease Model')
print('Accuracy score of the training data :', train_acc)
print('Accuracy score of the test data     :', test_acc)

Kidney Disease Model
Accuracy score of the training data : 0.975
Accuracy score of the test data     : 0.925


In [27]:
# ---------------------------------------------------------------------------
# Save model
# ---------------------------------------------------------------------------
filename = 'kidney_disease_model.sav'
pickle.dump(classifier, open(filename, 'wb'))
print(f"Saved model to {filename}")

Saved model to kidney_disease_model.sav


In [28]:
# quick sanity check reload
loaded_model = pickle.load(open(filename, 'rb'))
sample = np.asarray(X_test.iloc[0]).reshape(1, -1)
print('Sample prediction:', loaded_model.predict(sample), 'true label:', Y_test.iloc[0])

Sample prediction: [0] true label: 0


C:\Users\Riz\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
